<a href="https://colab.research.google.com/github/Darshika2004/cross-platform-spam-detector/blob/main/Spam_Detection_Web_Appipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Mount Google Drive to access the saved model files
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Install Hugging Face Transformers, PyTorch, and Gradio for the Web App UI
!pip install transformers torch gradio

In [3]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Define the directory path where the BERT model is saved in Google Drive
MODEL_PATH = "/content/drive/MyDrive/spam_detection/models/bert_model"

# Load tokenizer and model from the saved directory
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

# Set model to evaluation mode
model.eval()

print("Model and Tokenizer loaded successfully!")

Loading weights:   0%|          | 0/201 [00:02<?, ?it/s]

Model and Tokenizer loaded successfully!


In [10]:
import re
import gradio as gr
import torch

# Simple Keyword Matching for Highlighting
SUSPICIOUS_KEYWORDS = [
    "FREE",
    "Winner",
    "Crypto",
    "Lottery",
    "Click Here",
    "Click Link",
    "Urgent",
    "Claim",
]
URL_PATTERN = r"(https?://[^\s]+|www\.[^\s]+|bit\.ly[^\s]+|goo\.gl[^\s]+)"

SAFETY_TIPS = {
    "Email": [
        "📧 Check the sender's email address domain carefully.",
        "📧 Never click on unexpected links or attachments.",
        "📧 Be cautious of urgent wording demanding immediate action.",
    ],
    "SMS": [
        "📱 Never reply to unknown numbers with personal info.",
        "📱 Avoid clicking short links (e.g., bit.ly, goo.gl) from unknown texts.",
        "📱 Banks usually don't ask for sensitive information via SMS.",
    ],
    "YouTube": [
        "🎥 Beware of comment section links promising free giveaways or crypto.",
        "🎥 Verify if the channel has a verified badge.",
        "🎥 Report scam links using YouTube's built-in reporting tool.",
    ],
}


def analyze_message(text, platform):
    if not text.strip():
        return (
            "<div style='padding:15px; background-color:#334155 !important; color:#f8fafc !important; border-radius:8px;'><b>Please enter a message to analyze.</b></div>",
            "",
        )

    # 1. Automatic URL Detection & Keyword Highlighting (Both Links and Keywords)
    detected_urls = re.findall(URL_PATTERN, text, re.IGNORECASE)
    highlighted_text = text
    found_keywords = []

    # Highlight Keywords
    for kw in SUSPICIOUS_KEYWORDS:
        if re.search(re.escape(kw), highlighted_text, re.IGNORECASE):
            found_keywords.append(kw)
            highlighted_text = re.sub(
                f"({re.escape(kw)})",
                r"<mark style='background-color: #fef08a !important; color: #0f172a !important; padding: 2px 4px; border-radius: 4px; font-weight: bold;'>\1</mark>",
                highlighted_text,
                flags=re.IGNORECASE,
            )

    # Highlight Detected URLs / Links
    if detected_urls:
        for url in set(detected_urls):
            highlighted_text = highlighted_text.replace(
                url,
                f"<mark style='background-color: #fef08a !important; color: #0f172a !important; padding: 2px 4px; border-radius: 4px; font-weight: bold;'>{url}</mark>",
            )

    # 2. Model Prediction & AI Confidence Score
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, padding=True, max_length=128
    )
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.nn.functional.softmax(
            outputs.logits, dim=-1
        ).squeeze()

    ham_prob = float(probabilities[0]) * 100
    spam_prob = float(probabilities[1]) * 100

    is_spam = spam_prob > ham_prob
    confidence = spam_prob if is_spam else ham_prob

    # 3. Explanation
    explanation_reasons = []
    if is_spam:
        if detected_urls:
            explanation_reasons.append(
                f"Contains suspicious link(s): <code style='color:#dc2626 !important; background:#f1f5f9 !important; padding:2px 4px; border-radius:4px;'>{', '.join(detected_urls)}</code>"
            )
        if found_keywords:
            explanation_reasons.append(
                f"Triggered suspicious keywords: <b>{', '.join(found_keywords)}</b>"
            )
        if not explanation_reasons:
            explanation_reasons.append(
                "The message structure matches known spam patterns analyzed by the BERT model."
            )

    # 4. Color-Coded Cards with FIXED Contrast for Light & Dark Mode
    if is_spam:
        bg_card = "#fee2e2"
        border_card = "#ef4444"
        text_title = "#991b1b"
        text_body = "#7f1d1d"
        status_label = "🚨 Spam Alert (Not Secure)"
        confidence_label = f"{confidence:.1f}% Probability of Spam"
    else:
        bg_card = "#dcfce7"
        border_card = "#22c55e"
        text_title = "#166534"
        text_body = "#14532d"
        status_label = "✅ Safe Message (Ham)"
        confidence_label = f"{confidence:.1f}% Probability of Safe Message"

    result_html = f"""
    <div style='background-color: {bg_card} !important; border-left: 6px solid {border_card} !important; padding: 18px; border-radius: 8px;'>
        <h2 style='color: {text_title} !important; margin-top: 0;'>{status_label}</h2>

        <p style='font-size: 16px; color: {text_body} !important; font-weight: 600; margin-bottom: 12px;'>
            Model Confidence: <span style='font-size: 18px; font-weight: bold; color: {text_title} !important;'>{confidence_label}</span>
        </p>

        <!-- Model Comparison / Probability Bar Section -->
        <div style='margin-top: 12px; margin-bottom: 12px; background-color: #ffffff !important; padding: 12px; border-radius: 6px; color: #0f172a !important; border: 1px solid #cbd5e1;'>
            <b style='color: #1e293b !important; font-size: 14px;'>📊 Model Class Probabilities:</b>
            <div style='display: flex; justify-content: space-between; margin-top: 8px; font-size: 14px; font-weight: bold;'>
                <span style='color: #166534 !important;'>Ham Probability: {ham_prob:.1f}%</span>
                <span style='color: #991b1b !important;'>Spam Probability: {spam_prob:.1f}%</span>
            </div>
            <div style='width: 100%; background-color: #e2e8f0 !important; height: 10px; border-radius: 5px; margin-top: 6px; overflow: hidden; display: flex;'>
                <div style='width: {ham_prob}%; background-color: #22c55e !important; height: 100%;'></div>
                <div style='width: {spam_prob}%; background-color: #ef4444 !important; height: 100%;'></div>
            </div>
        </div>

        <!-- Analyzed Text Box -->
        <div style='margin-top: 10px; background-color: #ffffff !important; color: #0f172a !important; padding: 12px; border-radius: 6px; border: 1px solid #cbd5e1;'>
            <b style='color: {text_title} !important;'>Analyzed Text with Highlights:</b><br>
            <div style='margin-top: 5px; line-height: 1.5; color: #0f172a !important;'>{highlighted_text}</div>
        </div>
    </div>
    """

    tips = SAFETY_TIPS.get(platform, [])
    tips_html = (
        f"<div style='margin-top: 10px;'>"
        f"<h4 style='margin-bottom: 8px;'>🛡️ Safety Tips for {platform}:</h4>"
        f"<ul style='padding-left: 20px;'>"
        + "".join([f"<li style='margin-bottom: 4px;'>{tip}</li>" for tip in tips])
        + "</ul></div>"
    )

    return result_html, tips_html


# Launch modern Gradio UI Layout
with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="blue"), title="AI Spam & Threat Detector"
) as demo:
    gr.Markdown(
        """
        # 🛡️ Cross-Platform Spam Detector
        ### Secure Web Interface powered by Fine-Tuned BERT Model
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            platform_dropdown = gr.Dropdown(
                choices=["Email", "SMS", "YouTube"],
                value="Email",
                label="Select Message Platform",
            )
            input_text = gr.Textbox(
                lines=5,
                placeholder="Paste your message/email/comment here...",
                label="Message Text",
            )

            analyze_btn = gr.Button("🔍 Analyze Message", variant="primary")
            clear_btn = gr.ClearButton(components=[input_text])

        with gr.Column(scale=1):
            result_output = gr.HTML(label="Classification Result")
            tips_output = gr.HTML(label="Safety Tips")

    analyze_btn.click(
        fn=analyze_message,
        inputs=[input_text, platform_dropdown],
        outputs=[result_output, tips_output],
    )

demo.launch(share=True)

/tmp/ipykernel_1968/302314889.py:158: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f6588e385c17ef30b9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
